In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
viz_4244_from_joined_autofix_and_plots.py
-----------------------------------------
Section 4.2.4.4 assets generator (OH–FP bidirectional analysis).

Inputs
------
- Table_4.2.4.4_bidirectional_joined.csv  (required)
- Table_4.2.4.4_bidirectional_correlations.csv (optional; will be recomputed if missing/empty)

Outputs (English labels only)
-----------------------------
- bidirectional_summary/Table_4.2.4.4_bidirectional_correlations.csv  (Pearson/Spearman both)
- bidirectional_summary/figs_from_joined/Fig_4244_A_correlation_{pearson|spearman}.{png,pdf}
- bidirectional_summary/figs_from_joined/Fig_4244_B_scatter_{yvar}.{png,pdf}
- bidirectional_summary/figs_from_joined/Fig_4244_C_box_{metric}.{png,pdf}
- bidirectional_summary/Table_4.2.4.4_summary.csv
- bidirectional_summary/Table_4.2.4.4_stats.csv

Notes
-----
- If 'origin' column is absent, it is created as "unknown".
- JS divergence is "lower is better". For comparability, an inverted score
  JS_divergence_rev = 1 - pair_mean_JS_divergence is added for plots/tables.
- The script is defensive to missing columns; unavailable plots are skipped with a note.
"""

from __future__ import annotations
import argparse
from pathlib import Path
import sys
import warnings
import math

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# -----------------------------
# Utilities
# -----------------------------

FIGSIZE_HM = (9, 7)
FIGSIZE_SC = (9, 7)
FIGSIZE_BX = (9, 7)
DPI = 300

POSSIBLE_X = ["ARI_mean", "ARI_FP", "ARI", "ARI_OHFP"]  # x-axis candidate (OH->FP)
Y_CANDIDATES = [
    ("pair_mean_cosine_similarity", "Pair cosine similarity (FP→OH)"),
    ("FPcluster_median_cohesive_ratio", "FP cluster cohesion (FP→OH)"),
    ("pair_mean_JS_divergence", "Pair JS divergence (lower=better; FP→OH)"),
    ("JS_divergence_rev", "Pair JS divergence (rev., higher=better; FP→OH)"),
]
BOX_METRICS = [
    ("ARI_mean", "ARI (OH→FP)"),
    ("pair_mean_cosine_similarity", "Pair cosine similarity (FP→OH)"),
    ("FPcluster_median_cohesive_ratio", "FP cluster cohesion (FP→OH)"),
    ("JS_divergence_rev", "Pair JS divergence (rev., higher=better; FP→OH)"),
]

def find_joined(base: Path) -> Path | None:
    # glob for the canonical joined file
    pat = "**/paper_4.2.4.4_OHFP*/paper_4.2.4.4_OHFP/bidirectional_summary/Table_4.2.4.4_bidirectional_joined.csv"
    cand = list(base.glob(pat))
    if cand:
        # prefer the deepest (longest) path (latest project)
        cand = sorted(cand, key=lambda p: len(str(p)))
        return cand[-1]
    return None

def ensure_dirs(joined_path: Path) -> tuple[Path, Path]:
    summary_dir = joined_path.parent
    figs_dir = summary_dir / "figs_from_joined"
    figs_dir.mkdir(parents=True, exist_ok=True)
    return summary_dir, figs_dir

def read_csv_safe(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    # strip whitespace from columns for robustness
    df.columns = [c.strip() for c in df.columns]
    return df

def has_nonempty_corr(corr_path: Path) -> bool:
    if not corr_path.exists():
        return False
    try:
        df = pd.read_csv(corr_path)
    except Exception:
        return False
    return (df.shape[0] > 0) and (df.shape[1] > 0)

def pick_ari_column(df: pd.DataFrame) -> str | None:
    for c in POSSIBLE_X:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            return c
    return None

def prepare_df(df: pd.DataFrame) -> pd.DataFrame:
    # origin column
    if "origin" not in df.columns:
        df["origin"] = "unknown"
    # mode/index defaults
    if "mode" not in df.columns:
        df["mode"] = "NA"
    if "index" not in df.columns:
        df["index"] = "NA"
    # JS reversed score
    if "pair_mean_JS_divergence" in df.columns:
        js = df["pair_mean_JS_divergence"]
        # Normalize to [0,1] if out of range; otherwise just (1 - js)
        if js.min() < 0 or js.max() > 1:
            # try min-max normalization to [0,1], then reverse
            js_min, js_max = js.min(), js.max()
            if js_max > js_min:
                js01 = (js - js_min) / (js_max - js_min)
            else:
                js01 = pd.Series(0.5, index=js.index)
            df["JS_divergence_rev"] = 1 - js01
        else:
            df["JS_divergence_rev"] = 1 - js
    return df

def save_correlation_tables(df: pd.DataFrame, out_csv: Path, variables: list[str]) -> dict[str, pd.DataFrame]:
    d = df[variables].copy()
    d = d.dropna(axis=0, how="any")
    pear = d.corr(method="pearson")
    spear = d.corr(method="spearman")
    # Save long form with method column for convenience
    pear_long = pear.reset_index().melt(id_vars="index", var_name="variable", value_name="value")
    pear_long.insert(0, "method", "pearson")
    spear_long = spear.reset_index().melt(id_vars="index", var_name="variable", value_name="value")
    spear_long.insert(0, "method", "spearman")
    corr_long = pd.concat([pear_long, spear_long], ignore_index=True)
    corr_long.to_csv(out_csv, index=False)
    return {"pearson": pear, "spearman": spear}

def _save(fig, path_png: Path):
    fig.savefig(path_png, dpi=DPI, bbox_inches="tight")
    fig.savefig(path_png.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)

def plot_heatmap(corr: pd.DataFrame, title: str, outpath: Path):
    fig, ax = plt.subplots(figsize=FIGSIZE_HM)
    sns.heatmap(corr, annot=True, fmt=".2f", square=True, ax=ax, cbar=True)
    ax.set_title(title)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    _save(fig, outpath)

def plot_scatter_facets(df: pd.DataFrame, x: str, y: str, y_label: str, outpath: Path):
    # Facet by NbClust index; marker by mode; hue by origin
    if y not in df.columns or x not in df.columns:
        return
    d = df[[x, y, "origin", "mode", "index"]].dropna()
    if d.empty:
        return
    # main scatter
    g = sns.FacetGrid(d, col="index", hue="origin", style="mode", height=4, aspect=1.1, col_wrap=3, sharex=True, sharey=False)
    g.map_dataframe(sns.scatterplot, x=x, y=y, alpha=0.8)
    g.add_legend(title="Origin / Mode")
    g.set_axis_labels(x, y_label)
    g.set_titles(col_template="{col_name}")
    # overlay origin-wise means with 95% CI (overall, not per facet, to avoid small-n)
    means = d.groupby("origin")[[x, y]].agg(["mean", "std", "count"])
    # Compute 95% CI using t distribution; visualize as text in the super title
    txt_lines = []
    for org, row in means.iterrows():
        xm, xs, xn = row[(x, "mean")], row[(x, "std")], int(row[(x, "count")])
        ym, ys, yn = row[(y, "mean")], row[(y, "std")], int(row[(y, "count")])
        def ci(m, s, n):
            if n <= 1 or s is None or np.isnan(s):
                return (np.nan, np.nan)
            se = s / math.sqrt(n)
            tval = stats.t.ppf(0.975, df=max(n-1,1))
            return (m - tval*se, m + tval*se)
        xci = ci(xm, xs, xn); yci = ci(ym, ys, yn)
        txt_lines.append(f"{org}: x̄={xm:.3f} [{xci[0]:.3f},{xci[1]:.3f}], ȳ={ym:.3f} [{yci[0]:.3f},{yci[1]:.3f}] (n={xn})")
    g.fig.subplots_adjust(top=0.85)
    g.fig.suptitle(f"{x} vs {y_label}\nOrigin-wise mean ±95% CI  |  " + "  •  ".join(txt_lines), fontsize=10)
    _save(g.fig, outpath)

def plot_box_by_origin(df: pd.DataFrame, metric: str, label: str, outpath: Path):
    if metric not in df.columns:
        return
    d = df[[metric, "origin"]].dropna()
    if d.empty:
        return
    fig, ax = plt.subplots(figsize=FIGSIZE_BX)
    sns.boxplot(data=d, x="origin", y=metric, ax=ax)
    sns.stripplot(data=d, x="origin", y=metric, ax=ax, alpha=0.35, dodge=True)
    ax.set_xlabel("Origin")
    ax.set_ylabel(label)
    ax.set_title(f"{label} by Origin")
    _save(fig, outpath)

def zscore_series(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / (s.std(ddof=1) if s.std(ddof=1) not in (0, np.nan) else 1)

def one_sample_t(x: pd.Series) -> tuple[float, float]:
    # H0: mean(x) == 0
    if len(x) < 2:
        return (np.nan, np.nan)
    t, p = stats.ttest_1samp(x, popmean=0.0, nan_policy="omit")
    return t, p

def cohens_d_one_sample(x: pd.Series) -> float:
    # mean(x) / sd(x)
    sd = x.std(ddof=1)
    if not np.isfinite(sd) or sd == 0:
        return np.nan
    return x.mean() / sd

def welch_t_two_ind(x: pd.Series, y: pd.Series) -> tuple[float, float]:
    if len(x.dropna()) < 2 or len(y.dropna()) < 2:
        return (np.nan, np.nan)
    t, p = stats.ttest_ind(x, y, equal_var=False, nan_policy="omit")
    return t, p

def effect_size_cohens_d_ind(x: pd.Series, y: pd.Series) -> float:
    # pooled SD with unequal n (Glass not used here)
    nx, ny = x.count(), y.count()
    sx, sy = x.std(ddof=1), y.std(ddof=1)
    if nx < 2 or ny < 2 or not np.isfinite(sx) or not np.isfinite(sy):
        return np.nan
    sp = math.sqrt(((nx-1)*sx*sx + (ny-1)*sy*sy) / (nx + ny - 2)) if (nx+ny-2) > 0 else np.nan
    if not np.isfinite(sp) or sp == 0:
        return np.nan
    return (x.mean() - y.mean()) / sp

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", required=True, help="Base directory (project root under which joined is located)")
    ap.add_argument("--joined", default=None, help="Optional explicit path to joined CSV")
    args, _ = ap.parse_known_args()

    base = Path(args.base).resolve()
    joined = Path(args.joined).resolve() if args.joined else find_joined(base)
    if not joined or not joined.exists():
        print("ERROR: joined CSV not found. Use --joined to specify explicitly.", file=sys.stderr)
        sys.exit(2)

    summary_dir, figs_dir = ensure_dirs(joined)
    print("Joined CSV:", joined)
    print("Summary dir:", summary_dir)
    print("Figs dir   :", figs_dir)

    df = read_csv_safe(joined)
    df = prepare_df(df)

    # choose ARI column
    xcol = pick_ari_column(df)
    if xcol is None:
        # fallback: create ARI_mean from ARI_FP if available, else abort
        if "ARI_FP" in df.columns:
            df["ARI_mean"] = df["ARI_FP"]
            xcol = "ARI_mean"
        else:
            print("ERROR: No ARI-like column found among", POSSIBLE_X, file=sys.stderr)
            sys.exit(3)

    # variables for correlation / will filter to those present
    corr_vars = [xcol, "pair_mean_cosine_similarity", "pair_mean_JS_divergence",
                 "JS_divergence_rev", "FPcluster_median_cohesive_ratio",
                 "mean_OH_purity", "mean_OH_entropy"]
    corr_vars = [c for c in corr_vars if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
    if len(corr_vars) < 2:
        print("WARNING: Too few numeric variables for correlations:", corr_vars, file=sys.stderr)

    # (Re)compute correlations if missing or empty
    corr_csv = summary_dir / "Table_4.2.4.4_bidirectional_correlations.csv"
    corr_dict = {}
    if not has_nonempty_corr(corr_csv):
        if len(corr_vars) >= 2:
            corr_dict = save_correlation_tables(df, corr_csv, corr_vars)
            print("Correlation table recomputed and saved ->", corr_csv)
        else:
            print("Correlation table not computed due to insufficient variables.", file=sys.stderr)
    else:
        # Also compute in-memory for plotting
        try:
            # prefer to recompute in-memory for safety
            if len(corr_vars) >= 2:
                corr_dict = save_correlation_tables(df, corr_csv, corr_vars)
                print("Correlation table refreshed ->", corr_csv)
        except Exception as e:
            print("WARNING: could not refresh correlation tables:", e, file=sys.stderr)

    # ---- Plots: Heatmaps
    if corr_dict:
        pear = corr_dict.get("pearson")
        spear = corr_dict.get("spearman")
        if pear is not None and pear.shape[0] > 0:
            plot_heatmap(pear, "Correlation (Pearson)", figs_dir / "Fig_4244_A_correlation_pearson.png")
        if spear is not None and spear.shape[0] > 0:
            plot_heatmap(spear, "Correlation (Spearman)", figs_dir / "Fig_4244_A_correlation_spearman.png")

    # ---- Plots: Scatter (x = ARI_mean-like)
    for y, label in Y_CANDIDATES:
        if y in df.columns or (y == "JS_divergence_rev" and "pair_mean_JS_divergence" in df.columns):
            plot_scatter_facets(df, xcol, y, label, figs_dir / f"Fig_4244_B_scatter_{y}.png")

    # ---- Plots: Boxplots by origin
    for m, label in BOX_METRICS:
        if m in df.columns:
            plot_box_by_origin(df, m, label, figs_dir / f"Fig_4244_C_box_{m}.png")

    # ---- Tables: Origin-wise summary
    metrics_for_summary = [c for c, _ in BOX_METRICS if c in df.columns]
    group_cols = ["origin"]
    agg = {}
    for m in metrics_for_summary:
        agg[m] = ["mean", "std", "count"]
    summary = df.groupby(group_cols).agg(agg)
    # flatten columns
    summary.columns = [f"{m}_{stat}" for m, stat in summary.columns]
    summary = summary.reset_index()
    # add n_runs
    summary["n_runs"] = summary[[f"{metrics_for_summary[0]}_count"]].values if metrics_for_summary else 0
    # Save summary
    summary_out = summary_dir / "Table_4.2.4.4_summary.csv"
    summary.to_csv(summary_out, index=False)
    print("Saved:", summary_out)

    # ---- Stats: Directional contrast per origin
    # We compare OH→FP (xcol) vs each FP→OH metric, **after z-scoring within each origin**
    rows = []
    for org, sub in df.groupby("origin"):
        # ensure numeric and non-empty
        x = sub[xcol].dropna()
        if len(x) < 2:
            continue
        # z-score within origin for comparability
        z = sub.copy()
        for c in metrics_for_summary + [xcol]:
            if c in z.columns and pd.api.types.is_numeric_dtype(z[c]):
                z[c] = zscore_series(z[c].astype(float))
        for m in metrics_for_summary:
            if m == xcol or m not in z.columns:
                continue
            diff = z[xcol] - z[m]  # positive => OH→FP stronger than FP→OH on standardized scale
            t1, p1 = one_sample_t(diff)
            d1 = cohens_d_one_sample(diff)
            # For reference, also Welch t-test on raw scales (not standardized)
            t2, p2 = welch_t_two_ind(sub[xcol], sub[m])
            d2 = effect_size_cohens_d_ind(sub[xcol], sub[m])
            rows.append({
                "origin": org,
                "metric_FPtoOH": m,
                "n_diff": int(diff.count()),
                "mean_diff_z": float(diff.mean()),
                "t_one_sample": float(t1) if np.isfinite(t1) else np.nan,
                "p_one_sample": float(p1) if np.isfinite(p1) else np.nan,
                "cohens_d_one_sample": float(d1) if np.isfinite(d1) else np.nan,
                "t_welch_raw": float(t2) if np.isfinite(t2) else np.nan,
                "p_welch_raw": float(p2) if np.isfinite(p2) else np.nan,
                "cohens_d_ind_raw": float(d2) if np.isfinite(d2) else np.nan,
                "mean_OHtoFP_raw": float(sub[xcol].mean()),
                f"mean_{m}_raw": float(sub[m].mean()) if m in sub.columns else np.nan
            })
    stats_df = pd.DataFrame(rows)
    # BH correction within each metric across origins
    if not stats_df.empty:
        stats_df["p_BH_one_sample"] = np.nan
        stats_df["p_BH_welch_raw"] = np.nan
        for m in stats_df["metric_FPtoOH"].unique():
            mask = stats_df["metric_FPtoOH"] == m
            # BH for one-sample test
            ps = stats_df.loc[mask, "p_one_sample"].values
            order = np.argsort(ps)
            mK = np.sum(~np.isnan(ps))
            adj = np.full_like(ps, np.nan, dtype=float)
            rank = 1
            for idx in order:
                if not np.isfinite(ps[idx]):
                    continue
                adj_val = ps[idx] * mK / rank
                adj_val = min(adj_val, 1.0)
                adj[idx] = adj_val
                rank += 1
            stats_df.loc[mask, "p_BH_one_sample"] = adj
            # BH for Welch raw
            ps = stats_df.loc[mask, "p_welch_raw"].values
            order = np.argsort(ps)
            mK = np.sum(~np.isnan(ps))
            adj = np.full_like(ps, np.nan, dtype=float)
            rank = 1
            for idx in order:
                if not np.isfinite(ps[idx]):
                    continue
                adj_val = ps[idx] * mK / rank
                adj_val = min(adj_val, 1.0)
                adj[idx] = adj_val
                rank += 1
            stats_df.loc[mask, "p_BH_welch_raw"] = adj

        stats_out = summary_dir / "Table_4.2.4.4_stats.csv"
        stats_df.to_csv(stats_out, index=False)
        print("Saved:", stats_out)
    else:
        print("WARNING: No stats rows produced (check data/columns).", file=sys.stderr)

    print("\n✅ Done. Figures ->", figs_dir)
    print("   Tables  ->", summary_dir)

if __name__ == "__main__":
    warnings.filterwarnings("ignore")
    main()
    

usage: ipykernel_launcher.py [-h] --base BASE [--joined JOINED]
ipykernel_launcher.py: error: the following arguments are required: --base


SystemExit: 2